Homework 5
====================
ASEN 5264 Decision Making Under Uncertainty  
Cavlin Henggeler


In [1]:
using POMDPs
using DMUStudent.HW6
using POMDPTools: transition_matrices, reward_vectors, SparseCat, Deterministic, RolloutSimulator, DiscreteBelief, FunctionPolicy, ordered_states, ordered_actions, DiscreteUpdater
using QuickPOMDPs: QuickPOMDP
using POMDPModels: TigerPOMDP, TIGER_LEFT, TIGER_RIGHT, TIGER_LISTEN, TIGER_OPEN_LEFT, TIGER_OPEN_RIGHT
using NativeSARSOP: SARSOPSolver
using POMDPTesting: has_consistent_distributions

ArgumentError: ArgumentError: Package NativeSARSOP not found in current path.
- Run `import Pkg; Pkg.add("NativeSARSOP")` to install the NativeSARSOP package.

### Problem 1

In [ ]:
# ----------------------------------
# Updater: Rejection Particle Filter
# ----------------------------------

struct HW6Updater{M<:POMDP} <: Updater
    m::M
    n_particles::Int
end

function POMDPs.update(up::HW6Updater, b::DiscreteBelief, a, o)
    #bp_vec = zeros(length(states(up.m)))        # initialize vector of next beleif b'
    # bp = statetype(m)[]     # initialize new belief structure similar to the old?
    # Note that the ordering of the entries in bp_vec must be consistent with stateindex(m, s) 
    #(the container returned by states(m) does not necessarily obey this order)
    #states = ordered_states(up.m)

    # states = similar(b.states)
    #states = ordered_states(up.m)              
    bp_vec = zeros(length(states(up.m)))        # initialize vector of next beleif b'. represents states
    
    # simulate particles
    i = 1
    while i <= length(bp_vec)
        s = rand(b)
        sp, o_gen = @gen(:sp, :o)(up.m, s, a)    # Genertive model for s' and o
        if o_gen == o
            # update belief
            bp_vec[i] = sp 
            i += 1
        end
    end 

    # return a descrete belief b' represeting s'
    return DiscreteBelief(up.m, bp_vec) # im not sure im using this correctly
end

#-------
# Policy
#-------

struct HW6AlphaVectorPolicy{A} <: Policy
    alphas::Vector{Vector{Float64}} # alpha vectors
    alpha_actions::Vector{A}        # actions associated with alpha vectors
end

function POMDPs.action(p::HW6AlphaVectorPolicy, b::DiscreteBelief)
    
    beliefvec(b::DiscreteBelief) = b.b

    i = argmax([a.*b.b for a in p.alphas])

    return alpha_actions[i] # return the action associated with the best alpha vector

end

#------
# QMDP
#------

function qmdp_solve(m, discount=discount(m))

    # Fill in Value Iteration to compute the Q-values
    # "QMDP (algorithm 21.2) constructs a single alpha vector αa for each action 'a' using value iteration."

    acts = actiontype(m)[]
    alphas = Vector{Float64}[]

    function alpha_update(m, alphas, a, discount, S)
        alpha_p = [R(s,a) + discount*sum(pdf(transition(m,s,a),s)*maximum(ap[j] for alpha_p in alphas) for (j,sp) in enumerate(S)) for ap in actions(m)]
        return alpha_p
    end

    # For iterations 
    function alpha_vector_iteration(m, alphas, discount, S; k_iters=1)
        for k in 1:k_iters
            # update alpha vectors
            alphas = update(m, alphas, a, discount, S)
        end
        return alphas
    end

    S = ordered_states(m)               # ordered states of the POMDP
    a_vec = zeros(length(S))               # initialize vector of next beleif b'. represents states


    for a in actions(m)
    
        a_vec = zeros(length(S))               # initialize vector of next beleif b'. represents states
    
        push!(acts, a)
        push!(alphas, a_vec)

    end

    return HW6AlphaVectorPolicy(alphas, acts)
end


# function alphavector_iteration(𝒫::POMDP, M, Γ)
#     for k in 1:M.k_max
#         Γ = update(𝒫, M, Γ)
#     end
#     return Γ
# end
# 
# struct QMDP
#     k_max # maximum number of iterations
# end
#     
# function update(𝒫::POMDP, M::QMDP, Γ)
#     𝒮, 𝒜, R, T, γ = 𝒫.𝒮, 𝒫.𝒜, 𝒫.R, 𝒫.T, 𝒫.γ
#     Γ′ = [[R(s,a) + γ*sum(T(s,a,s′)*maximum(α′[j] for α′ in Γ)for (j,s′) in enumerate(𝒮)) for s in 𝒮] for a in 𝒜]
#     return Γ′
# end
# 
# function solve(M::QMDP, 𝒫::POMDP)
#     Γ = [zeros(length(𝒫.𝒮)) for a in 𝒫.𝒜]
#     Γ = alphavector_iteration(𝒫, M, Γ)
#     return AlphaVectorPolicy(𝒫, Γ, 𝒫.𝒜)
# end
    


In [ ]:
m = TigerPOMDP()

qmdp_p = qmdp_solve(m)
# Note: you can use the QMDP.jl package to verify that your QMDP alpha vectors are correct.
sarsop_p = solve(SARSOPSolver(), m)
up = HW6Updater(m)

@show mean(simulate(RolloutSimulator(max_steps=500), m, qmdp_p, up) for _ in 1:5000)
@show mean(simulate(RolloutSimulator(max_steps=500), m, sarsop_p, up) for _ in 1:5000)